# Step3 — Análisis histórico descriptivo (Ecuador, 2015–2025)

**Proyecto:** EFCI Ecuador (ENSO Flood Chronicity Index)
**Fase:** 3 — Análisis histórico descriptivo por provincia y cantón
**Autor:** Jerson Achicanoy (FAO OER) · implementación asistida
**Fecha:** 2026-08-05

**Objetivo de este notebook:** calcular, por provincia y por cantón (y por
tipo de peligro dentro de cada uno), los indicadores históricos
descriptivos definidos en el diseño (Fase 0 §9): número de eventos,
años/meses afectados, frecuencia por categoría ENSO (ICEN y ONI por
separado), estadísticos de impacto (media/mediana/máximo/desviación
estándar), y tendencia temporal (Mann-Kendall + pendiente de Sen).

**No calcula** los índices compuestos (Severidad, Recurrencia, Persistencia,
EFCI) — eso es Fase 4–8. Esta fase produce los insumos descriptivos que esos
índices normalizarán y ponderarán.

**Decisiones metodológicas (justificadas en `docs/methodology/03_historical_analysis.md`):**
- Todo se calcula **por tipo de peligro por separado** (`hazard_type`) y
  también **combinado** (fila `hazard_type = "TODOS"`) — no se promedia a
  ciegas entre inundación/deslizamiento/aluvión/socavamiento.
- La frecuencia ENSO se reporta por **ICEN y por ONI por separado** — es una
  decisión de diseño abierta (Fase 0 §19.4); con datos reales ya se puede
  comparar antes de fijar un solo índice ancla.
- Tendencia con **Mann-Kendall + pendiente de Sen** (no regresión lineal):
  apropiado para series de conteo no necesariamente normales, y no asume
  linealidad.
- Se usan los 21,743 eventos completos (Fase 2) — ninguno se excluye por
  `canton_confidence`, todos tienen un `canton_code` reportado válido.

Notebook comentado línea por línea (regla del proyecto 2026-08-05). Lógica
inline; `src/utilidades` solo aporta helpers transversales.

**English:** Historical Analysis

Descriptive statistics by province and canton, per hazard type and combined: recurrence and persistence, impact statistics, ENSO-category frequency and temporal trend (Mann-Kendall with Sen's slope).

Outputs: `hist_*` tables in `data/processed/`.

## 0. Configuración del entorno

**English:** 0. Setup

In [ ]:
# Fase 3 tampoco requiere GEE ni CDS: trabaja sobre la salida de la Fase 2.

# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent  # notebook vive en <repo>/notebooks/
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Tabular data handling
import pandas as pd  # tablas
# Numerical operations
import numpy as np  # operaciones numéricas
# Mann-Kendall trend test
import pymannkendall as mk  # prueba de tendencia Mann-Kendall + pendiente de Sen
# Efficient reader for GeoPackage layers
import pyogrio  # lectura eficiente de hazard_event_geo.gpkg (solo atributos, sin geometría)

# Show up to 40 columns when printing DataFrames
pd.set_option("display.max_columns", 40)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)

# Create a logger that writes to the logs folder
logger = get_logger("step3_historical_analysis", config, log_dir=PROJECT_ROOT / "logs")
logger.info("Fase 3 iniciada — configuración cargada desde %s", CONFIG_PATH)

# First year of the analysis window
YEAR_MIN = config["date_range"]["start_year"]  # 2015
# Last year of the analysis window
YEAR_MAX = config["date_range"]["end_year"]  # 2025
# List of all years in the analysis window
ALL_YEARS = list(range(YEAR_MIN, YEAR_MAX + 1))  # rango completo, para no perder años sin eventos

## 1. Carga de datos

Se usa `hazard_event_geo.gpkg` (Fase 2) — ya trae `canton_code`/`province_code`
verificados, más `icen_category`/`oni_category` de la Fase 1. Se lee solo la
tabla de atributos (`read_geometry=False`): esta fase no necesita geometría,
solo agregación tabular.

Se incorpora/reemplaza `canton_name_official` y `province_name_official`
desde la capa `adm2_canton` de `admin_boundaries.gpkg`, mediante un join por
**atributo** sobre `canton_code` (no espacial). Dos razones: (1) evita
agrupar por el texto libre `canton`/`province` de SNGRE, que tiene variantes
de escritura en 9 provincias (Fase 2 §4.3) y duplicaría grupos indebidamente;
(2) la versión de `province_name_official` que trae Fase 2 viene del cruce
espacial punto-en-polígono y por eso hereda sus 35 nulos (eventos sin
polígono contenedor) — la versión por atributo, keyed on `canton_code`
(100% de cobertura), no tiene ese problema.

**English:** 1. Load data

In [ ]:
# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Load the hazard events table without geometry
he = pyogrio.read_dataframe(processed_dir / "hazard_event_geo.gpkg", read_geometry=False)
# Log the number of events loaded
logger.info("hazard_event_geo cargado: %d filas", len(he))

# he ya trae "province_name_official" de la Fase 2, pero esa versión viene
# del cruce ESPACIAL punto-en-polígono, así que hereda sus 35 nulos (los
# eventos sin polígono contenedor, Fase 2 §4.4) — y no trae en absoluto
# "canton_name_official". Se reemplaza/incorpora ambas aquí con un join por
# ATRIBUTO sobre canton_code contra la capa adm2_canton (no espacial):
# canton_code es la clave 100%-completa y autoritativa reportada por SNGRE
# (Fase 2 §4.4-4.8), así que el nombre oficial que le corresponde es el de
# ese código en el mapa oficial, no el del polígono geométrico que contiene
# el punto (que puede diferir, o directamente no existir, en los casos ya
# documentados en Fase 2).
# Load the official canton and province names
adm2 = pyogrio.read_dataframe(
    processed_dir / "admin_boundaries.gpkg", layer="adm2_canton", read_geometry=False
)[["canton_code", "canton_name_official", "province_name_official"]].drop_duplicates()
# Remember the row count to detect accidental changes
n_antes = len(he)
# Attach official canton and province names to each event
he = he.drop(columns=["province_name_official"]).merge(adm2, on="canton_code", how="left")
# Check the name join neither duplicates nor loses events
assert len(he) == n_antes, "el join adm2 no debe duplicar ni perder filas"
# Count events without an official canton or province name
n_sin_nombre = he[["canton_name_official", "province_name_official"]].isna().any(axis=1).sum()
# Warn only if some events lack a name
if n_sin_nombre:
    # Log a warning so the correction is auditable
    logger.warning(
        "%d eventos sin nombre oficial de cantón/provincia tras el join por "
        "canton_code (canton_code no presente en admin_boundaries adm2_canton)",
        n_sin_nombre,
    )
# Log the coverage check
logger.info(
    "canton_name_official/province_name_official incorporados por atributo "
    "(canton_code, 100%% de cobertura); sin match: %d", n_sin_nombre
)

# Variables de impacto que se resumen en la sección 3 — subconjunto de las
# ~20 disponibles, elegido por relevancia directa a severidad (Fase 0 §10.1):
# población, vivienda, educación, salud, agricultura, ganadería.
# Impact variables to summarize
IMPACT_VARS = [
    "deaths", "affected_direct", "affected_indirect", "sheltered",
    "houses_affected", "houses_destroyed", "schools_affected",
    "health_facilities_affected", "crops_affected_ha", "livestock_affected",
]

## 2. Indicadores de recurrencia y persistencia temporal por unidad administrativa

Para cada provincia y cada cantón (y cada combinación con `hazard_type`, más
la fila agregada `"TODOS"`): número de eventos, años afectados, meses
afectados (total de combinaciones año-mes con al menos un evento — no solo
el mes calendario 1-12), primer y último evento registrado.

**English:** 2. Recurrence and temporal persistence by administrative unit

In [ ]:
# Recurrence and persistence indicators per administrative unit
def resumen_recurrencia(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Calcula indicadores de recurrencia y persistencia por unidad administrativa.

    Se ejecuta dos veces sobre el mismo DataFrame de entrada: una vez con
    `hazard_type` incluido en `group_cols` (desglose por peligro) y otra sin
    él pero añadiendo una fila sintética 'TODOS' (combinado) — ver sección 5.

    Args:
        df: hazard_event_geo, ya filtrado si corresponde.
        group_cols: columnas de agrupación (p. ej. ['province_code','province']
            o ['province_code','province','canton_code','canton']).

    Returns:
        Un registro por combinación de group_cols con: n_eventos,
        anios_afectados, meses_afectados, primer_evento, ultimo_evento.

    English:
    Summarize recurrence and persistence per administrative unit.

    Args:
        df: Hazard events with year and month columns.
        group_cols: Columns that define the unit.

    Returns:
        Table with event count, years, months and days affected per unit.
    """
    # Group events by the requested administrative columns
    g = df.groupby(group_cols)
    # Count events and distinct years, months and days affected
    out = g.agg(
        n_eventos=("event_id", "count"),
        anios_afectados=("year", "nunique"),
        primer_evento=("event_date", "min"),
        ultimo_evento=("event_date", "max"),
    ).reset_index()
    # meses_afectados = combinaciones únicas (año, mes) con al menos un evento
    # — no es lo mismo que "cuántos de los 12 meses del año" (estacionalidad);
    # esa métrica se calcula aparte en la sección 4.
    # Count the distinct year-month combinations per unit
    meses = df.groupby(group_cols).apply(
        lambda d: d[["year", "month"]].drop_duplicates().shape[0], include_groups=False
    ).rename("meses_afectados").reset_index()
    # Attach the month count to the summary
    out = out.merge(meses, on=group_cols, how="left")
    # Return the per-unit table
    return out

## 3. Estadísticos de impacto (media, mediana, máximo, desviación estándar)

Se reporta también `n_eventos` junto a cada estadístico: un máximo o una
media calculados sobre pocos eventos no deben leerse con la misma confianza
que uno calculado sobre cientos — se deja explícito, no implícito.

**English:** 3. Impact statistics (mean, median, maximum, standard deviation)

In [ ]:
# Impact statistics per unit and impact variable
def resumen_impactos(df: pd.DataFrame, group_cols: list[str], impact_vars: list[str]) -> pd.DataFrame:
    """Calcula media/mediana/máximo/desviación estándar de cada variable de
    impacto, por unidad administrativa, en formato largo (una fila por
    combinación de grupo + variable de impacto).

    Formato largo (no ancho) a propósito: con 10 variables x 4 estadísticos
    serían 40 columnas en formato ancho, difícil de leer y de graficar; en
    formato largo se puede filtrar/pivotar según necesidad en fases futuras.

    Args:
        df: hazard_event_geo, ya filtrado si corresponde.
        group_cols: columnas de agrupación.
        impact_vars: nombres de columnas de impacto a resumir.

    Returns:
        DataFrame largo: group_cols + ['variable_impacto','n','media','mediana','maximo','desv_std'].

    English:
    Summarize impact variables (mean, median, maximum, standard deviation, count) per unit.

    Args:
        df: Hazard events.
        group_cols: Columns that define the unit.
        impact_vars: Impact columns to summarize.

    Returns:
        Long table with one row per unit and impact variable.
    """
    # List that collects one table per variable
    rows = []
    # Loop over the impact variables
    for var in impact_vars:  # una variable de impacto a la vez, evita un pivot gigante
        # Mean, median, maximum, standard deviation and count of the variable per unit
        g = df.groupby(group_cols)[var].agg(
            n="count", media="mean", mediana="median", maximo="max", desv_std="std"
        ).reset_index()
        # Record which variable the table describes
        g["variable_impacto"] = var  # etiqueta de qué variable es esta fila
        # Store the table
        rows.append(g)
    # Stack the per-variable tables
    result = pd.concat(rows, ignore_index=True)
    result["desv_std"] = result["desv_std"].fillna(0)  # std de 1 solo dato es NaN, no 0 "real" — se documenta en el técnico
    # Return the final table
    return result

## 4. Frecuencia por categoría ENSO (ICEN y ONI por separado)

Conteo de eventos por unidad administrativa y por categoría ENSO, calculado
independientemente para ICEN (índice costero, fuente ENFEN Perú) y ONI
(índice oceánico, fuente NOAA) — se reportan ambos, no se colapsa a un solo
"índice ancla" todavía (decisión abierta, Fase 0 §19.4).

**English:** 4. Frequency by ENSO category (ICEN and ONI separately)

In [ ]:
# Event counts by ENSO category for each index
def resumen_enso(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Cuenta eventos por categoría ENSO (ICEN y ONI), por unidad administrativa.

    Args:
        df: hazard_event_geo, ya filtrado si corresponde.
        group_cols: columnas de agrupación.

    Returns:
        DataFrame largo: group_cols + ['indice_enso' (ICEN/ONI), 'categoria', 'n_eventos'].

    English:
    Count events by ENSO category, for ICEN and ONI separately.

    Args:
        df: Hazard events with ICEN and ONI category columns.
        group_cols: Columns that define the unit.

    Returns:
        Table with event counts per unit, index and category.
    """
    # Events per unit and ICEN category
    icen = df.groupby(group_cols + ["icen_category"]).size().rename("n_eventos").reset_index()
    # Standardize the category column name
    icen = icen.rename(columns={"icen_category": "categoria"})
    # Label the rows as ICEN
    icen["indice_enso"] = "ICEN"

    # Events per unit and ONI category
    oni = df.groupby(group_cols + ["oni_category"]).size().rename("n_eventos").reset_index()
    # Standardize the category column name
    oni = oni.rename(columns={"oni_category": "categoria"})
    # Label the rows as ONI
    oni["indice_enso"] = "ONI"

    # Stack both indices
    return pd.concat([icen, oni], ignore_index=True)

## 5. Tendencia temporal (Mann-Kendall + pendiente de Sen)

Se calcula sobre la serie de conteo de eventos por año, **reindexada al
rango completo 2015–2025** (los años sin eventos cuentan como 0 — omitirlos
sesgaría la prueba). Se requieren al menos 4 años con datos para que la
prueba tenga sentido mínimo; por debajo de eso se marca explícitamente como
`datos_insuficientes` en vez de reportar un p-valor engañoso.

**English:** 5. Temporal trend (Mann-Kendall and Sen's slope)

In [ ]:
# Annual-count trend per unit (Mann-Kendall test and Sen's slope)
def tendencia_mann_kendall(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Aplica Mann-Kendall + pendiente de Sen a la serie anual de eventos
    de cada unidad administrativa.

    Args:
        df: hazard_event_geo, ya filtrado si corresponde.
        group_cols: columnas de agrupación.

    Returns:
        DataFrame: group_cols + ['tendencia','p_valor','pendiente_sen','anios_con_dato'].
        'tendencia' es 'datos_insuficientes' si hay menos de 4 años con al
        menos un evento (umbral mínimo convencional para Mann-Kendall).

    English:
    Test the annual event-count trend per unit with Mann-Kendall and Sen's slope.

    Args:
        df: Hazard events with a year column.
        group_cols: Columns that define the unit.

    Returns:
        Table with trend direction, p-value, slope and years with data per unit.
    """
    # Minimum number of years with events to run the test
    MIN_ANIOS = 4  # por debajo de esto, Mann-Kendall no es confiable — se documenta, no se oculta
    # List that collects one row per unit
    results = []
    # Loop over each unit
    for keys, sub in df.groupby(group_cols):
        # Make the group key a tuple
        keys = keys if isinstance(keys, tuple) else (keys,)  # normalizar a tupla siempre
        # Annual event counts, with zero for years without events
        annual = sub.groupby("year").size().reindex(ALL_YEARS, fill_value=0)  # 0 explícito en años sin evento
        # Number of years that have at least one event
        n_anios_con_dato = (annual > 0).sum()  # años con al menos 1 evento
        # Skip units with too few years
        if n_anios_con_dato < MIN_ANIOS:
            # Record 'insufficient data' for those units
            results.append((*keys, "datos_insuficientes", np.nan, np.nan, n_anios_con_dato))
            # Move on to the next unit
            continue
        # Run the Mann-Kendall test
        test = mk.original_test(annual.values)  # prueba no paramétrica, no asume normalidad/linealidad
        # Record the trend, p-value, Sen's slope and number of years
        results.append((*keys, test.trend, round(test.p, 4), round(test.slope, 4), n_anios_con_dato))
    # Column names of the result
    cols = group_cols + ["tendencia", "p_valor", "pendiente_sen", "anios_con_dato"]
    # Return the trend table
    return pd.DataFrame(results, columns=cols)

## 6. Ejecución: nivel cantón y nivel provincia, desglosado y combinado

Se corre cada función dos veces (cantón, provincia) y dos veces más dentro
de cada nivel (por `hazard_type` y combinado `"TODOS"`), para no perder la
segmentación por tipo de peligro (Fase 0 §17) mientras se conserva también
la vista agregada que pide el diseño original.

Se agrupa por `province_code`/`canton_code` + el nombre **oficial** (Fase 2,
INEC), no por el texto libre `province`/`canton` de SNGRE — 9 provincias
tienen más de una variante de escritura en ese texto libre, lo que
duplicaría grupos indebidamente si se usara como llave de agrupación.

**English:** 6. Run: canton and province level, per hazard and combined

In [ ]:
# Compute all indicator tables for one administrative level
def ejecutar_nivel(df: pd.DataFrame, admin_cols: list[str]) -> dict[str, pd.DataFrame]:
    """Ejecuta las 4 funciones de resumen (recurrencia, impactos, ENSO,
    tendencia) para un nivel administrativo dado, en dos vistas: desglosada
    por hazard_type y combinada ('TODOS').

    Args:
        df: hazard_event_geo completo.
        admin_cols: columnas del nivel administrativo (cantón o provincia).

    Returns:
        Diccionario con las 4 tablas finales (concatenando desglosado + combinado).

    English:
    Compute recurrence, impact, ENSO and trend tables for one administrative level.

    Args:
        df: Hazard events with official names.
        admin_cols: Columns that define the administrative level.

    Returns:
        Dictionary of tables, each stacking the per-hazard and the combined views.
    """
    # Grouping columns for the per-hazard view
    group_por_hazard = admin_cols + ["hazard_type"]

    # --- vista desglosada por tipo de peligro ---
    # Recurrence per hazard
    recurrencia_h = resumen_recurrencia(df, group_por_hazard)
    # Impact statistics per hazard
    impactos_h = resumen_impactos(df, group_por_hazard, IMPACT_VARS)
    # ENSO frequency per hazard
    enso_h = resumen_enso(df, group_por_hazard)
    # Trend per hazard
    tendencia_h = tendencia_mann_kendall(df, group_por_hazard)

    # --- vista combinada: todos los hazard_type juntos, etiquetada 'TODOS' ---
    # Copy the events for the combined view
    df_todos = df.copy()
    # Label every event as 'TODOS' (all hazards)
    df_todos["hazard_type"] = "TODOS"  # sobrescribe solo para esta agregación combinada
    # Recurrence for all hazards combined
    recurrencia_t = resumen_recurrencia(df_todos, group_por_hazard)
    # Impact statistics for all hazards combined
    impactos_t = resumen_impactos(df_todos, group_por_hazard, IMPACT_VARS)
    # ENSO frequency for all hazards combined
    enso_t = resumen_enso(df_todos, group_por_hazard)
    # Trend for all hazards combined
    tendencia_t = tendencia_mann_kendall(df_todos, group_por_hazard)

    # Return all tables, stacking per-hazard and combined views
    return {
        "recurrencia": pd.concat([recurrencia_h, recurrencia_t], ignore_index=True),
        "impactos": pd.concat([impactos_h, impactos_t], ignore_index=True),
        "enso": pd.concat([enso_h, enso_t], ignore_index=True),
        "tendencia": pd.concat([tendencia_h, tendencia_t], ignore_index=True),
    }

# Se agrupa por CÓDIGO + nombre OFICIAL (Fase 2, INEC), no por el nombre de
# texto libre de SNGRE ('province'/'canton'): 9 provincias tienen más de una
# variante de escritura en el texto libre (tildes/mayúsculas, ver Fase 2
# §4.3), lo que duplicaría grupos si se agrupara por esas columnas. El
# nombre oficial es 1:1 con el código por construcción (viene de la propia
# capa de límites administrativos).
# Columns that identify a canton
CANTON_COLS = ["province_code", "province_name_official", "canton_code", "canton_name_official"]
# Columns that identify a province
PROVINCE_COLS = ["province_code", "province_name_official"]

# Log the start of the canton-level run
logger.info("Calculando indicadores a nivel cantón...")
# Compute all indicators at canton level
resultados_canton = ejecutar_nivel(he, CANTON_COLS)

# Log the start of the province-level run
logger.info("Calculando indicadores a nivel provincia...")
# Compute all indicators at province level
resultados_provincia = ejecutar_nivel(he, PROVINCE_COLS)

# Loop over the canton tables
for nombre, tabla in resultados_canton.items():
    # Log each canton table size
    logger.info("Cantón — tabla '%s': %d filas", nombre, len(tabla))
# Loop over the province tables
for nombre, tabla in resultados_provincia.items():
    # Log each province table size
    logger.info("Provincia — tabla '%s': %d filas", nombre, len(tabla))

## 7. Validación de resultados

**English:** 7. Validation

In [ ]:
# Chequeo 1: la suma de n_eventos de la vista 'TODOS' por cantón debe igualar
# el total de hazard_event_geo (21,743) — confirma que la agregación no
# perdió ni duplicó eventos.
# Total events counted at canton level in the combined view
total_todos_canton = resultados_canton["recurrencia"].loc[
    resultados_canton["recurrencia"]["hazard_type"] == "TODOS", "n_eventos"
].sum()
# Log the total against the expected number of events
logger.info("Suma n_eventos (vista TODOS, nivel cantón): %d (esperado %d)", total_todos_canton, len(he))
# Check the canton aggregation matches the total number of events
assert total_todos_canton == len(he), "La agregación por cantón no cuadra con el total de eventos"

# Total events counted at province level in the combined view
total_todos_provincia = resultados_provincia["recurrencia"].loc[
    resultados_provincia["recurrencia"]["hazard_type"] == "TODOS", "n_eventos"
].sum()
# Check the province aggregation matches the total number of events
assert total_todos_provincia == len(he), "La agregación por provincia no cuadra con el total de eventos"

# Chequeo 2: todo cantón/provincia con eventos debe tener al menos una fila
# de tendencia (aunque sea 'datos_insuficientes') — nada se cae silenciosamente.
# Number of cantons with events
n_cantones_con_eventos = he.groupby(CANTON_COLS).ngroups
# Number of cantons in the trend table
n_cantones_en_tendencia = resultados_canton["tendencia"]["canton_code"].nunique()
# Log both numbers
logger.info("Cantones con eventos: %d | cantones en tabla de tendencia: %d", n_cantones_con_eventos, n_cantones_en_tendencia)
# Check no canton is missing from the trend table
assert n_cantones_con_eventos == n_cantones_en_tendencia

# Vista rápida: top 10 cantones por número de eventos (vista combinada)
# Ten cantons with the most events
top10 = resultados_canton["recurrencia"][resultados_canton["recurrencia"]["hazard_type"]=="TODOS"] \
    .sort_values("n_eventos", ascending=False).head(10)
# Print the top ten
print(top10[["province_name_official","canton_name_official","n_eventos","anios_afectados","meses_afectados"]].to_string(index=False))

# Distribución de tendencias a nivel cantón (vista combinada)
# Print a header for the trend distribution
print("\nDistribución de tendencias (cantón, TODOS los peligros):")
# Print how many cantons fall in each trend class
print(resultados_canton["tendencia"].loc[resultados_canton["tendencia"]["hazard_type"]=="TODOS", "tendencia"].value_counts())

## 8. Exportación

Se exportan 8 tablas a `data/processed/` (4 indicadores × 2 niveles
administrativos), en Parquet y CSV.

**English:** 8. Export

In [ ]:
# Create the output folder if needed
out_dir = ensure_dir(processed_dir)
# List of exported table names
written = []
# Loop over the two administrative levels
for nivel, resultados in [("canton", resultados_canton), ("provincia", resultados_provincia)]:
    # Loop over the tables of each level
    for nombre_tabla, df_tabla in resultados.items():
        # File base name
        base_name = f"hist_{nombre_tabla}_{nivel}"
        # Parquet path
        parquet_path = out_dir / f"{base_name}.parquet"
        # CSV path
        csv_path = out_dir / f"{base_name}.csv"
        # Write Parquet
        df_tabla.to_parquet(parquet_path, index=False)
        # Write CSV
        df_tabla.to_csv(csv_path, index=False, encoding="utf-8")
        # Remember the exported name
        written.append(base_name)
        # Log each exported table
        logger.info("Exportado: %s (%d filas)", base_name, len(df_tabla))

# Print the output folder
print("Tablas exportadas a", out_dir, ":")
# Loop over the exported names
for w in written:
    # Print each name
    print(" -", w)

## 9. Próximos pasos (Fase 4–8: índices)

1. Índice de Severidad (Fase 4): normalizar las variables de `hist_impactos_*`
   (método a decidir — min-max vs. z-score vs. percentil, Fase 0 §19.4).
2. Índice de Recurrencia (Fase 5): usar `hist_recurrencia_*` sobre el total
   de eventos ENSO en el periodo.
3. Índice de Persistencia (Fase 6): a partir de `meses_afectados` y de un
   cálculo adicional de meses *consecutivos* (no incluido en esta fase, que
   solo cuenta meses afectados en total, no rachas).
4. Índice de Magnitud ENSO (Fase 7): usar `hist_enso_*` para decidir el
   criterio de cohorte OR/AND (Fase 0 §10.6, H4) con datos reales.
5. Decidir el indicador ENSO ancla (ICEN vs. ONI vs. ambos) comparando
   `hist_enso_canton.csv` — ahora hay datos reales para esa discusión.
6. Revisar los cantones con `tendencia == "datos_insuficientes"` — son
   candidatos a un tratamiento distinto en el EFCI final (menor confianza).